# 10 minutes to VTL Engine

The [VTL Engine](https://github.com/Meaningful-Data/vtlengine) runs [VTL 2.1](https://sdmx.org/wp-content/uploads/VTL-2.1-Reference-Manual.pdf) scripts against tabular data: you bring a script, a description of your data, and the data itself, and the engine parses, type-checks, and executes it.

This notebook reproduces every example from the *10 minutes to VTL Engine* walkthrough, running entirely in your browser on the Pyodide kernel.

## Run

Use `run` when your data is in pandas DataFrames or CSV files.

### Example 1: Simple run

In [ ]:
import pandas as pd

from vtlengine import run

script = "DS_A <- DS_1 * 10;"
data_structures = {
    "datasets": [
        {
            "name": "DS_1",
            "DataStructure": [
                {"name": "Id_1", "type": "Integer", "role": "Identifier", "nullable": False},
                {"name": "Me_1", "type": "Number", "role": "Measure", "nullable": True},
            ],
        }
    ]
}
datapoints = {"DS_1": pd.DataFrame({"Id_1": [1, 2, 3], "Me_1": [10, 20, 30]})}

run(script=script, data_structures=data_structures, datapoints=datapoints)["DS_A"].data

### Example 2: Run with scalar values

`run` accepts scalar inputs through the `scalar_values` parameter.

In [ ]:
script = """
    DS_r <- DS_1[filter Me_1 = Sc_1];
    Sc_r <- Sc_1 + 10;
    Sc_r2 <- Sc_r * 2;
    Sc_r3 <- null;
"""
data_structures = {
    "datasets": [
        {
            "name": "DS_1",
            "DataStructure": [
                {"name": "Id_1", "type": "Integer", "role": "Identifier", "nullable": False},
                {"name": "Me_1", "type": "Number", "role": "Measure", "nullable": True},
            ],
        }
    ],
    "scalars": [{"name": "Sc_1", "type": "Number"}],
}
datapoints = {"DS_1": pd.DataFrame({"Id_1": [1, 2, 3], "Me_1": [10, 20, 30]})}

run(
    script=script,
    data_structures=data_structures,
    datapoints=datapoints,
    scalar_values={"Sc_1": 20},
    return_only_persistent=True,
)

## Run SDMX

Use `run_sdmx` when you work with SDMX data. The SDMX structure and data files for this example are bundled in the `data/` folder next to this notebook.

In [ ]:
from pathlib import Path

from pysdmx.io import get_datasets

from vtlengine import run_sdmx

datasets = get_datasets(Path("data/data.xml"), Path("data/metadata.xml"))

# Add a new measure Me_4 with the same value as OBS_VALUE.
run_sdmx("DS_r <- DS_1 [calc Me_4 := OBS_VALUE];", datasets)["DS_r"].data

## Semantic Analysis

`semantic_analysis` validates a script and infers the output data structures without executing it.

### Example 4: Correct VTL

In [ ]:
from vtlengine import semantic_analysis

script = "DS_A <- DS_1 * 10;"
data_structures = {
    "datasets": [
        {
            "name": "DS_1",
            "DataStructure": [
                {"name": "Id_1", "type": "Integer", "role": "Identifier", "nullable": False},
                {"name": "Me_1", "type": "Number", "role": "Measure", "nullable": True},
            ],
        }
    ]
}

semantic_analysis(script=script, data_structures=data_structures)

### Example 5: Incorrect VTL

The same script, but `Me_1` is declared as `String` instead of `Number`, so `semantic_analysis` raises a `SemanticError`.

In [ ]:
from vtlengine.Exceptions import SemanticError

bad_structures = {
    "datasets": [
        {
            "name": "DS_1",
            "DataStructure": [
                {"name": "Id_1", "type": "Integer", "role": "Identifier", "nullable": False},
                {"name": "Me_1", "type": "String", "role": "Measure", "nullable": True},
            ],
        }
    ]
}

try:
    semantic_analysis(script=script, data_structures=bad_structures)
except SemanticError as exc:
    print(f"{type(exc).__name__}: {exc}")

### Example 6: Using SDMX structures

`semantic_analysis` also accepts SDMX structure files or `pysdmx` objects. These snippets need real SDMX structure files, so they are shown for reference rather than executed here:

```python
from pathlib import Path

from vtlengine import semantic_analysis

# Using an SDMX-ML structure file
sa_result = semantic_analysis(script=script, data_structures=Path("path/to/structure.xml"))

# Using pysdmx objects directly
from pysdmx.io import read_sdmx

msg = read_sdmx(Path("path/to/structure.xml"))
dsds = msg.get_data_structure_definitions()
sa_result = semantic_analysis(script=script, data_structures=dsds)
```

## Prettify

`prettify` formats a VTL script to improve readability.

In [ ]:
from vtlengine import prettify

script = """
    define hierarchical ruleset accountingEntry (variable rule ACCOUNTING_ENTRY) is
        B = C - D errorcode "Balance (credit-debit)" errorlevel 4;
        N = A - L errorcode "Net (assets-liabilities)" errorlevel 4
    end hierarchical ruleset;

    DS_r <- check_hierarchy(BOP, accountingEntry rule ACCOUNTING_ENTRY dataset);
"""

print(prettify(script))

For the full API reference, see the [API documentation](https://docs.vtlengine.meaningfuldata.eu/latest/api.html).